# Stage 0 — Freeze shared micro resources

Build a deterministic 5K/2K subset from the frozen E1 Train/Val memberships and reuse the compatible image-level LCC SCRFD cache when it is attached. This notebook does not read CelebA held-out Test, train a model, or tune on LCC.



In [1]:
# Edit only these paths when automatic discovery is ambiguous.
E1_ARTIFACT_DIR = "/kaggle/input/datasets/locngodev/e1-v5-3-artifacts/E1_v5_3_mnv3_small"
CELEBA_ROOT = "/kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof"
LCC_FASD_ROOT = "/kaggle/input/datasets/aleksandrpikul222/lcc-fasd/LCC_FASD"
LCC_CACHE_DIR = "/kaggle/input/datasets/locngodev/smartface-pad-lcc-scrfd-cache-v1/smartface-pad-lcc-scrfd-cache-v1"  # Existing compatible manifest/cache/snapshot directory
SCRFD_MODEL_PATH = ""
OUTPUT_ROOT = "/kaggle/working/pad_micro_search_v1"
MICRO_TRAIN_N = 5000
MICRO_VAL_N = 2000
SEED = 42
BUILD_OPTIONAL_CROP_CACHE = False
OPTIONAL_CROP_CACHE_MAX_GB = 2.0

In [2]:
import csv
import hashlib
import importlib.util
import json
import math
import os
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path, PurePosixPath

import cv2
import numpy as np

E1_MANIFEST_NAME = "celeba_spoof_preliminary_v5_3_edge_mnv3_small_seed42.npz"
E1_RUN_CONFIG_NAME = "mnv3s_e1_preliminary_v5_3_edge_run_config.json"
E1_BBOX_CACHE_NAME = "celeba_scrfd_bbox_cache_v5_3_preliminary_seed42.json"
LCC_MANIFEST_NAME = "evaluation_unit_manifest.csv"
LCC_CACHE_NAME = "external_scrfd_cache.json"
LCC_SNAPSHOT_NAME = "cross_dataset_protocol_snapshot.json"
SCRFD_EXPECTED_SHA256 = "5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a"
OUTPUT_ROOT = str(Path(OUTPUT_ROOT))
STAGE0_DIR = Path(OUTPUT_ROOT) / "stage0"
if (MICRO_TRAIN_N, MICRO_VAL_N, SEED) != (5000, 2000, 42):
    raise RuntimeError("Frozen Stage 0 protocol requires exactly 5,000 Train, 2,000 Val, seed 42")
if STAGE0_DIR.exists():
    shutil.rmtree(STAGE0_DIR)
STAGE0_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(value):
    return hashlib.sha256(value.encode("utf-8")).hexdigest()

def split_fingerprint(keys):
    return sha256_text("\n".join(sorted(map(str, keys))))

def discover_exact(filename, search_root="/kaggle/input", explicit_dir=""):
    if explicit_dir:
        root = Path(explicit_dir).expanduser()
        if root.is_file() and root.name == filename:
            return root.resolve()
        if not root.is_dir():
            raise FileNotFoundError(f"Configured directory does not exist: {root}")
        matches = sorted(p.resolve() for p in root.rglob(filename) if p.is_file())
    else:
        root = Path(search_root)
        matches = sorted(p.resolve() for p in root.rglob(filename) if p.is_file()) if root.exists() else []
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one {filename}, found {len(matches)}. "
            f"Set an explicit directory/path. Matches: {[str(p) for p in matches[:10]]}"
        )
    return matches[0]

def discover_dataset_root(marker_relative, explicit_root, display_name):
    if explicit_root:
        root = Path(explicit_root).expanduser().resolve()
        if not (root / marker_relative).is_file():
            raise FileNotFoundError(f"{display_name} marker not found: {root / marker_relative}")
        return root
    matches = []
    search = Path("/kaggle/input")
    if search.exists():
        for p in search.rglob(Path(marker_relative).name):
            if p.is_file() and p.as_posix().endswith(marker_relative.replace("\\", "/")):
                root_index = len(PurePosixPath(marker_relative).parts) - 1
                matches.append(p.parents[root_index])
    matches = sorted(set(p.resolve() for p in matches))
    if len(matches) != 1:
        raise RuntimeError(
            f"Could not resolve one {display_name} root from {marker_relative}; "
            f"found {len(matches)}. Set its path in the top config cell."
        )
    return matches[0]

def key_subject(key):
    parts = PurePosixPath(str(key).replace("\\", "/")).parts
    for i, part in enumerate(parts[:-1]):
        if part.lower() in {"train", "test"}:
            return parts[i + 1]
    raise ValueError(f"Cannot derive E1 subject id from frozen key: {key}")

def label_of(meta_value):
    if len(meta_value) <= 40:
        raise ValueError("CelebA metadata item has no attack code at index 40")
    raw = int(meta_value[40])
    if raw == 0:
        return 0
    if raw in (1, 2, 3, 4, 5, 6, 7):
        return 1
    return 2

def largest_remainder_counts(labels, target_n):
    counts = np.bincount(np.asarray(labels, dtype=np.int64), minlength=3).astype(np.int64)
    if target_n > int(counts.sum()):
        raise ValueError(f"Requested {target_n} items from only {int(counts.sum())}")
    quotas = target_n * counts.astype(np.float64) / max(1, int(counts.sum()))
    allocation = np.floor(quotas).astype(np.int64)
    remaining = int(target_n - allocation.sum())
    order = sorted(range(3), key=lambda c: (-(quotas[c] - allocation[c]), c))
    for c in order[:remaining]:
        allocation[c] += 1
    if np.any(allocation > counts):
        raise AssertionError("Largest-remainder allocation exceeds source class count")
    return allocation.tolist(), counts.tolist()

def select_micro(keys, labels, target_n, seed):
    by_class = defaultdict(list)
    for key, label in zip(map(str, keys), map(int, labels)):
        by_class[label].append(key)
    allocation, source_counts = largest_remainder_counts(labels, target_n)
    selected = []
    for class_id, count in enumerate(allocation):
        ranked = sorted(
            by_class[class_id],
            key=lambda key: hashlib.sha256(f"{seed}|{class_id}|{key}".encode()).hexdigest()
        )
        selected.extend(ranked[:count])
    selected = sorted(selected)
    label_by_key = {str(k): int(v) for k, v in zip(keys, labels)}
    selected_labels = np.asarray([label_by_key[k] for k in selected], dtype=np.int64)
    return selected, selected_labels, allocation, source_counts



def ordered_path_fingerprint(paths):
    return sha256_text("\n".join(map(str, paths)))


CELEBA_BBOX_STATUS_FACTORS = {
    "VALID": 1.55,
    "FALLBACK_DET_FAIL": 1.50,
    "FALLBACK_SUSPICIOUS": 1.50,
}


def validate_celeba_bbox_record(record, key):
    if not isinstance(record, dict):
        raise TypeError("CelebA bbox record must be a mapping for {}".format(key))
    status = str(record.get("status", ""))
    if status == "INVALID":
        raise RuntimeError("Selected micro key has INVALID E1 bbox record: {}".format(key))
    if status not in CELEBA_BBOX_STATUS_FACTORS:
        raise RuntimeError("Unknown/unusable E1 bbox status {} for {}".format(status, key))
    if "crop_factor" not in record:
        raise KeyError("E1 bbox record is missing crop_factor for {}".format(key))
    try:
        crop_factor = float(record["crop_factor"])
    except (TypeError, ValueError) as exc:
        raise ValueError("Invalid E1 crop_factor for {}: {!r}".format(key, record.get("crop_factor"))) from exc
    expected = CELEBA_BBOX_STATUS_FACTORS[status]
    if not math.isfinite(crop_factor) or not math.isclose(crop_factor, expected, rel_tol=0.0, abs_tol=1e-6):
        raise ValueError(
            "E1 status/crop-factor mismatch for {}: status={} factor={} expected={}".format(
                key, status, crop_factor, expected
            )
        )
    bbox = np.asarray(record.get("bbox_xyxy", []), dtype=np.float64)
    if bbox.shape != (4,) or not np.isfinite(bbox).all() or bbox[2] <= bbox[0] or bbox[3] <= bbox[1]:
        raise ValueError("Invalid E1 bbox_xyxy for {}: {!r}".format(key, record.get("bbox_xyxy")))
    return status, crop_factor


def audit_selected_bbox_records(train_keys, val_keys, records):
    audit = {"overall": Counter(), "train": Counter(), "val": Counter(), "crop_factor_by_status": {}}
    for split_name, keys in (("train", train_keys), ("val", val_keys)):
        for key in map(str, keys):
            status, factor = validate_celeba_bbox_record(records[key], key)
            audit[split_name][status] += 1
            audit["overall"][status] += 1
            audit["crop_factor_by_status"].setdefault(status, Counter())[format(factor, ".2f")] += 1
    return {
        "overall_status_counts": dict(audit["overall"]),
        "train_status_counts": dict(audit["train"]),
        "val_status_counts": dict(audit["val"]),
        "crop_factor_counts_by_status": {
            status: dict(counts) for status, counts in audit["crop_factor_by_status"].items()
        },
        "accepted_status_factor_policy": CELEBA_BBOX_STATUS_FACTORS,
    }


def runtime_crop_bgr(image, bbox_xyxy, factor):
    h, w = image.shape[:2]
    x1, y1, x2, y2 = map(float, bbox_xyxy)
    bw, bh = x2 - x1, y2 - y1
    if bw <= 0 or bh <= 0:
        raise ValueError(f"Invalid xyxy bbox: {bbox_xyxy}")
    extent = max(bw, bh) * float(factor)
    side = int(extent)
    if side <= 0:
        raise ValueError(f"Invalid crop side: {side}")
    cx, cy = x1 + bw / 2.0, y1 + bh / 2.0
    xs, ys = int(cx - extent / 2.0), int(cy - extent / 2.0)
    xa, ya, xb, yb = max(0, xs), max(0, ys), min(w, xs + side), min(h, ys + side)
    if xb <= xa or yb <= ya:
        raise RuntimeError("Empty crop intersection")
    crop = image[ya:yb, xa:xb]
    return cv2.copyMakeBorder(
        crop, max(0, -ys), max(0, ys + side - h),
        max(0, -xs), max(0, xs + side - w), cv2.BORDER_REFLECT_101
    )

def resize_letterbox_rgb(rgb, size=224):
    h, w = rgb.shape[:2]
    ratio = size / max(h, w)
    nh, nw = max(1, int(h * ratio)), max(1, int(w * ratio))
    interp = cv2.INTER_LANCZOS4 if ratio > 1 else cv2.INTER_AREA
    image = cv2.resize(rgb, (nw, nh), interpolation=interp)
    dh, dw = size - nh, size - nw
    return cv2.copyMakeBorder(
        image, dh // 2, dh - dh // 2, dw // 2, dw - dw // 2, cv2.BORDER_REFLECT_101
    )

def exact_artifact_path(name, parent=""):
    return discover_exact(name, explicit_dir=parent)

print("Resolving frozen E1 resources...")
e1_manifest_path = exact_artifact_path(E1_MANIFEST_NAME, E1_ARTIFACT_DIR)
e1_run_config_path = exact_artifact_path(E1_RUN_CONFIG_NAME, E1_ARTIFACT_DIR)
e1_bbox_cache_path = exact_artifact_path(E1_BBOX_CACHE_NAME, E1_ARTIFACT_DIR)
celeba_root = discover_dataset_root("metas/intra_test/train_label.json", CELEBA_ROOT, "CelebA-Spoof")
train_label_path = celeba_root / "metas/intra_test/train_label.json"
if not train_label_path.is_file():
    raise FileNotFoundError(train_label_path)

# Read only the two frozen memberships below; no held-out membership is loaded.
parent_npz = np.load(e1_manifest_path, allow_pickle=False)
parent_train = parent_npz["train_keys"].astype(str)
parent_val = parent_npz["val_keys"].astype(str)
parent_npz.close()
run_config = json.loads(e1_run_config_path.read_text())
if run_config.get("run_mode") != "preliminary":
    raise RuntimeError(f"Expected frozen E1 preliminary mode, got {run_config.get('run_mode')!r}")
train_fp = split_fingerprint(parent_train)
val_fp = split_fingerprint(parent_val)
expected_fps = run_config.get("split_fingerprints", {})
if expected_fps.get("train") != train_fp or expected_fps.get("val") != val_fp:
    raise RuntimeError("Frozen E1 Train/Val fingerprints do not match the run config")
if set(parent_train) & set(parent_val):
    raise RuntimeError("Frozen E1 Train and Val keys overlap")
if {key_subject(k) for k in parent_train} & {key_subject(k) for k in parent_val}:
    raise RuntimeError("Frozen E1 Train and Val subjects overlap")

train_meta = json.loads(train_label_path.read_text())
missing_meta = [k for k in np.concatenate([parent_train, parent_val]) if str(k) not in train_meta]
if missing_meta:
    raise RuntimeError(f"{len(missing_meta)} frozen Train/Val keys are absent from train_label.json")
train_labels_full = np.asarray([label_of(train_meta[str(k)]) for k in parent_train], dtype=np.int64)
val_labels_full = np.asarray([label_of(train_meta[str(k)]) for k in parent_val], dtype=np.int64)
micro_train, micro_train_labels, train_alloc, train_source_counts = select_micro(
    parent_train, train_labels_full, MICRO_TRAIN_N, SEED
)
micro_val, micro_val_labels, val_alloc, val_source_counts = select_micro(
    parent_val, val_labels_full, MICRO_VAL_N, SEED
)
micro_train_codes = np.asarray([int(train_meta[k][40]) for k in micro_train], dtype=np.int64)
micro_val_codes = np.asarray([int(train_meta[k][40]) for k in micro_val], dtype=np.int64)
assert set(micro_train).issubset(set(parent_train))
assert set(micro_val).issubset(set(parent_val))
assert not (set(micro_train) & set(micro_val))
assert not ({key_subject(k) for k in micro_train} & {key_subject(k) for k in micro_val})

source_bbox_cache = json.loads(e1_bbox_cache_path.read_text())
bbox_records = source_bbox_cache.get("records", {})
selected_keys = micro_train + micro_val
missing_bbox = [k for k in selected_keys if k not in bbox_records]
if missing_bbox:
    raise RuntimeError(f"Frozen CelebA bbox cache misses {len(missing_bbox)} selected keys")
if len(micro_train) != 5000 or len(micro_val) != 2000:
    raise RuntimeError("Deterministic selection did not produce the frozen 5,000/2,000 sizes")
if set(map(int, micro_train_labels)) != {0, 1, 2} or set(map(int, micro_val_labels)) != {0, 1, 2}:
    raise RuntimeError("Each micro split must contain all three E1 classes 0/1/2")
bbox_status_audit = audit_selected_bbox_records(micro_train, micro_val, bbox_records)
missing_celeba_images = [str(celeba_root / k) for k in selected_keys if not (celeba_root / k).is_file()]
if missing_celeba_images:
    raise FileNotFoundError(
        "{} selected CelebA images do not resolve under {}. Examples: {}".format(
            len(missing_celeba_images), celeba_root, missing_celeba_images[:5]
        )
    )
print("Selected CelebA bbox status/factor audit:", json.dumps(bbox_status_audit, sort_keys=True))
print("Resolved all selected CelebA images:", len(selected_keys), "under", celeba_root)
micro_bbox_payload = {
    "schema": "celeba_micro_bbox_cache_v1",
    "source_file": e1_bbox_cache_path.name,
    "source_sha256": sha256_file(e1_bbox_cache_path),
    "policy": source_bbox_cache.get("policy", {}),
    "records": {k: bbox_records[k] for k in selected_keys},
}
bbox_out = STAGE0_DIR / "celeba_micro_bbox_cache.json"
bbox_out.write_text(json.dumps(micro_bbox_payload, indent=2, allow_nan=False) + "\n")

micro_npz_path = STAGE0_DIR / "micro_manifest_v1_seed42.npz"
np.savez_compressed(
    micro_npz_path,
    micro_train_keys=np.asarray(micro_train, dtype=str),
    micro_val_keys=np.asarray(micro_val, dtype=str),
    micro_train_labels=micro_train_labels,
    micro_val_labels=micro_val_labels,
    micro_train_attack_codes=micro_train_codes,
    micro_val_attack_codes=micro_val_codes,
)
micro_json = {
    "schema": "pad_micro_manifest_v1",
    "seed": int(SEED),
    "source_run_mode": run_config["run_mode"],
    "parent_e1_manifest_path": str(e1_manifest_path),
    "parent_e1_manifest_sha256": sha256_file(e1_manifest_path),
    "parent_train_fingerprint": train_fp,
    "parent_val_fingerprint": val_fp,
    "micro_train_n": len(micro_train),
    "micro_val_n": len(micro_val),
    "micro_train_class_counts": dict(Counter(map(int, micro_train_labels))),
    "micro_val_class_counts": dict(Counter(map(int, micro_val_labels))),
    "micro_train_allocation": train_alloc,
    "micro_val_allocation": val_alloc,
    "source_train_class_counts": train_source_counts,
    "source_val_class_counts": val_source_counts,
    "micro_train_keys_fingerprint": split_fingerprint(micro_train),
    "micro_val_keys_fingerprint": split_fingerprint(micro_val),
    "bbox_status_audit": bbox_status_audit,
    "overlap_checks": {
        "train_subset_of_parent_train": True,
        "val_subset_of_parent_val": True,
        "train_val_key_overlap_n": 0,
        "train_val_subject_overlap_n": 0,
    },
}
(STAGE0_DIR / "micro_manifest_v1_seed42.json").write_text(json.dumps(micro_json, indent=2, allow_nan=False) + "\n")
(STAGE0_DIR / "micro_train_keys.json").write_text(json.dumps(micro_train, indent=2, allow_nan=False) + "\n")
(STAGE0_DIR / "micro_val_keys.json").write_text(json.dumps(micro_val, indent=2, allow_nan=False) + "\n")

print("Frozen E1 manifest path/SHA:", e1_manifest_path, sha256_file(e1_manifest_path))
print("Frozen E1 Train/Val fingerprints:", train_fp, val_fp)
print("Frozen E1 source run mode:", run_config["run_mode"])
print("Frozen micro manifest:", len(micro_train), "Train /", len(micro_val), "Val")
print("Train classes:", micro_json["micro_train_class_counts"], "Val classes:", micro_json["micro_val_class_counts"])

# Reuse the existing external LCC manifest/cache only when snapshot policy and identities agree.
def find_lcc_artifact(name, required=True):
    if name in {LCC_MANIFEST_NAME, LCC_CACHE_NAME, LCC_SNAPSHOT_NAME} and LCC_CACHE_DIR:
        base = Path(LCC_CACHE_DIR).expanduser()
    elif name in {"CLIENT_TEST.txt", "IMPOSTER_TEST.txt"} and LCC_FASD_ROOT:
        base = Path(LCC_FASD_ROOT).expanduser()
    else:
        base = Path("/kaggle/input")
    matches = sorted(p.resolve() for p in base.rglob(name) if p.is_file()) if base.exists() else []
    if len(matches) > 1:
        raise RuntimeError(f"Found multiple {name} files; set LCC_CACHE_DIR. Matches: {[str(p) for p in matches[:8]]}")
    if not matches:
        if required:
            raise FileNotFoundError(f"Missing {name} under {base}")
        return None
    return matches[0]

def read_csv_rows(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

lcc_manifest_src = find_lcc_artifact(LCC_MANIFEST_NAME, required=False)
lcc_cache_src = find_lcc_artifact(LCC_CACHE_NAME, required=False)
lcc_snapshot_src = find_lcc_artifact(LCC_SNAPSHOT_NAME, required=False)
if lcc_manifest_src is not None:
    src_manifest_rows = read_csv_rows(lcc_manifest_src)
else:
    client_list = find_lcc_artifact("CLIENT_TEST.txt", required=False)
    imposter_list = find_lcc_artifact("IMPOSTER_TEST.txt", required=False)
    if client_list is None or imposter_list is None:
        raise RuntimeError(
            "No LCC evaluation manifest/cache bundle found. Attach the compatible Stage 0 LCC "
            "resource bundle, or provide both CLIENT_TEST.txt and IMPOSTER_TEST.txt plus SCRFD."
        )
    src_manifest_rows = []
    for label, list_path in [(0, client_list), (1, imposter_list)]:
        for raw_line in list_path.read_text().splitlines():
            rel = raw_line.strip().split()[0] if raw_line.strip() else ""
            if rel:
                src_manifest_rows.append({
                    "path": rel.replace("\\", "/"), "label": label, "partition": "evaluation",
                    "dataset": "lcc_fasd", "subject_id": "", "attack_subtype": "unknown",
                    "file_sha256": "",
                })
    if not src_manifest_rows:
        raise RuntimeError("LCC client/imposter lists are empty")
if not src_manifest_rows:
    raise RuntimeError("LCC evaluation manifest is empty")
source_lcc_paths = [str(row["path"]).replace("\\", "/") for row in src_manifest_rows]
if len(source_lcc_paths) != len(set(source_lcc_paths)):
    raise RuntimeError("LCC source evaluation manifest has duplicate candidate paths")
for row in src_manifest_rows:
    if int(row["label"]) not in (0, 1):
        raise RuntimeError("LCC source labels must remain 0=Real, 1=Attack: {}".format(row))

def resolve_lcc_root(rows):
    if LCC_FASD_ROOT:
        root = Path(LCC_FASD_ROOT).expanduser().resolve()
        if not root.is_dir():
            raise FileNotFoundError(root)
        return root
    rel_parts = PurePosixPath(str(rows[0]["path"]).replace("\\", "/")).parts
    matches = []
    search = Path("/kaggle/input")
    if search.exists():
        for p in search.rglob(rel_parts[-1]):
            if p.is_file() and tuple(p.parts[-len(rel_parts):]) == rel_parts:
                matches.append(p.resolve().parents[len(rel_parts) - 1])
    matches = sorted(set(matches))
    if len(matches) != 1:
        raise RuntimeError(
            f"Cannot uniquely resolve LCC root from {rows[0]['path']}; "
            "set LCC_FASD_ROOT explicitly."
        )
    return matches[0]

lcc_root = resolve_lcc_root(src_manifest_rows)
print("LCC source manifest/cache/snapshot:", lcc_manifest_src, lcc_cache_src, lcc_snapshot_src)
print("Resolved LCC root:", lcc_root)
compatible = False
src_records = {}
cache_policy = {}
src_cache = {}
if lcc_cache_src is not None and lcc_snapshot_src is not None and lcc_manifest_src is not None:
    candidate_cache = json.loads(lcc_cache_src.read_text())
    candidate_snapshot = json.loads(lcc_snapshot_src.read_text())
    candidate_policy = candidate_cache.get("policy", {})
    snapshot_policy = candidate_snapshot.get("cache_policy", {})
    compatible = (
        candidate_policy == snapshot_policy
        and candidate_policy.get("schema") == "external_scrfd_v1"
        and candidate_policy.get("detector_sha256") == SCRFD_EXPECTED_SHA256
        and candidate_cache.get("sampled_manifest_sha256") == candidate_snapshot.get("sampled_manifest_sha256")
    )
    if compatible:
        src_cache = candidate_cache
        src_records = src_cache.get("records", {})
        cache_policy = candidate_policy
        print("Reusing compatible frozen LCC SCRFD cache; detector will not be rerun.")
    else:
        print("Attached LCC cache is incompatible with the frozen policy; rebuild requested below.")

if not compatible:
    required_insightface_version = None
    if lcc_snapshot_src is not None:
        rebuild_snapshot = json.loads(lcc_snapshot_src.read_text())
        required_insightface_version = rebuild_snapshot.get("cache_policy", {}).get("insightface")
    missing_packages = [
        name for name in ("insightface", "onnxruntime")
        if importlib.util.find_spec(name) is None
    ]
    if missing_packages:
        version_pin = (
            "insightface=={}".format(required_insightface_version)
            if required_insightface_version else "insightface"
        )
        raise RuntimeError(
            "LCC cache rebuild is required, but Kaggle is missing {}. Install before rerunning "
            "Stage 0 with: %pip install -q {} onnxruntime. A compatible attached cache avoids "
            "these dependencies and remains preferred.".format(missing_packages, version_pin)
        )
    try:
        import insightface
        from insightface import model_zoo
        import onnxruntime as ort
    except Exception as exc:
        raise RuntimeError(
            "LCC cache rebuild needs working InsightFace and ONNX Runtime. In a Kaggle setup "
            "cell run: %pip install -q insightface onnxruntime, then rerun Stage 0."
        ) from exc
    insightface_version = str(getattr(insightface, "__version__", "unknown"))
    if required_insightface_version and insightface_version != str(required_insightface_version):
        raise RuntimeError(
            "LCC cache rebuild InsightFace version mismatch: frozen snapshot requires {}, "
            "runtime has {}. Install the snapshot version before rebuilding.".format(
                required_insightface_version, insightface_version
            )
        )
    if "CPUExecutionProvider" not in ort.get_available_providers():
        raise RuntimeError("LCC SCRFD cache rebuild requires ONNX Runtime CPUExecutionProvider")
    if SCRFD_MODEL_PATH:
        model_candidate = Path(SCRFD_MODEL_PATH).expanduser()
        if model_candidate.is_dir():
            model_path = discover_exact("det_500m.onnx", explicit_dir=str(model_candidate))
        elif model_candidate.is_file():
            model_path = model_candidate.resolve()
        else:
            raise FileNotFoundError(model_candidate)
    else:
        model_path = discover_exact("det_500m.onnx")
    detector_sha = sha256_file(model_path)
    if detector_sha != SCRFD_EXPECTED_SHA256:
        raise RuntimeError(f"SCRFD SHA mismatch: expected {SCRFD_EXPECTED_SHA256}, got {detector_sha}")
    print("SCRFD rebuild model path/SHA:", model_path, detector_sha)
    detector = model_zoo.get_model(str(model_path), providers=["CPUExecutionProvider"])
    if detector is None:
        raise RuntimeError(f"InsightFace could not load SCRFD model: {model_path}")
    expected_paths = [str(row["path"]).replace("\\", "/") for row in src_manifest_rows]
    pending = {
        path: {"status": "PENDING", "path": path, "dataset": "lcc_fasd", "label": int(row["label"]),
               "frame_id": sha256_text(f"lcc_fasd|{path}|0"), "crop_factor": 1.55}
        for path, row in zip(expected_paths, src_manifest_rows)
    }
    size_attempts = [(640, 640), (480, 480), (320, 320)]
    for size in size_attempts:
        detector.prepare(ctx_id=-1, input_size=size, det_thresh=0.5)
        for path in expected_paths:
            if pending[path]["status"] != "PENDING":
                continue
            image_path = lcc_root / path
            image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            if image is None:
                pending[path]["status"] = "MISSING_IMAGE"
                pending[path]["error"] = "cv2.imread returned None"
                continue
            detections, _landmarks = detector.detect(image, max_num=0, metric="default")
            detections = np.asarray(detections) if detections is not None else np.empty((0, 5))
            valid = []
            for det in detections:
                x1, y1, x2, y2, conf = map(float, det[:5])
                if x2 > x1 and y2 > y1:
                    area = (x2 - x1) * (y2 - y1)
                    valid.append((area, conf, (x1, y1, x2, y2)))
            if valid:
                valid.sort(key=lambda item: (-item[0], -item[1], item[2][0], item[2][1], item[2][2], item[2][3]))
                _area, confidence, box = valid[0]
                pending[path].update({
                    "status": "VALID", "bbox_xyxy": list(box), "confidence": confidence,
                    "n_faces": len(valid), "ambiguous_multiface": len(valid) > 1,
                    "det_input_size": list(size),
                    "image_width": int(image.shape[1]), "image_height": int(image.shape[0]),
                })
    for path in expected_paths:
        if pending[path]["status"] == "PENDING":
            pending[path].update({"status": "NO_VALID_FACE", "n_faces": 0, "ambiguous_multiface": False})
    src_records = {record["frame_id"]: record for record in pending.values()}
    try:
        import insightface
        insightface_version = getattr(insightface, "__version__", "unknown")
    except Exception:
        insightface_version = "unknown"
    cache_policy = {
        "schema": "external_scrfd_v1",
        "detector_sha256": detector_sha,
        "sizes": [list(x) for x in size_attempts],
        "confidence": 0.5,
        "crop_factor": 1.55,
        "selection": "largest valid face; confidence then xyxy tie-break",
        "sampling": {
            "frames_per_video": 16,
            "seed": 42,
            "method": "uniform np.linspace integer indices",
        },
        "min_valid_frames": 1,
        "opencv": cv2.__version__,
        "insightface": insightface_version,
    }
    sampled_manifest_sha = sha256_text("\n".join(
        f"{row['path']}|{int(row['label'])}" for row in src_manifest_rows
    ))
    src_cache = {
        "policy": cache_policy, "sampled_manifest_sha256": sampled_manifest_sha,
        "records": src_records,
    }
    print("Built LCC SCRFD cache once from the supplied evaluation manifest/list files.")

record_by_path = defaultdict(list)
record_by_id = {}
for record in src_records.values():
    record_by_path[str(record.get("path", ""))].append(record)
    frame_id = str(record.get("frame_id", ""))
    if not frame_id or frame_id in record_by_id:
        raise RuntimeError("LCC source cache has a missing or duplicate record frame_id")
    record_by_id[frame_id] = record
source_path_set = set(source_lcc_paths)
if set(record_by_path) != source_path_set:
    missing_paths = sorted(source_path_set - set(record_by_path))
    extra_paths = sorted(set(record_by_path) - source_path_set)
    raise RuntimeError(
        "LCC cache candidate paths differ from the source manifest; missing={}, extra={}".format(
            missing_paths[:5], extra_paths[:5]
        )
    )
lcc_rows = []
seen_lcc_paths = set()
missing_lcc_records = []
missing_lcc_images = []
for row in src_manifest_rows:
    rel = str(row["path"]).replace("\\", "/")
    if rel in seen_lcc_paths:
        raise RuntimeError(f"Duplicate LCC evaluation path: {rel}")
    seen_lcc_paths.add(rel)
    label = int(row["label"])
    if label not in (0, 1):
        raise RuntimeError(f"Unexpected LCC label {label} for {rel}")
    image_path = lcc_root / rel
    if not image_path.is_file():
        missing_lcc_images.append(str(image_path))
    matches = record_by_path.get(rel, [])
    if len(matches) != 1:
        missing_lcc_records.append((rel, len(matches)))
        continue
    record = matches[0]
    if int(record.get("label", -1)) != label:
        raise RuntimeError(f"LCC label/cache mismatch for {rel}")
    lcc_rows.append({
        "path": rel, "label": label, "partition": row.get("partition", "evaluation"),
        "dataset": row.get("dataset", "lcc_fasd"), "subject_id": row.get("subject_id", ""),
        "attack_subtype": row.get("attack_subtype", "unknown"),
        "file_sha256": row.get("file_sha256", ""), "record_id": str(record["frame_id"]),
        "status": record.get("status", "UNKNOWN"),
        "n_faces": int(record.get("n_faces", 0)),
        "ambiguous_multiface": bool(record.get("ambiguous_multiface", False)),
    })
if missing_lcc_images:
    raise FileNotFoundError(
        f"{len(missing_lcc_images)} LCC paths do not resolve under {lcc_root}; "
        f"examples: {missing_lcc_images[:5]}. Set LCC_FASD_ROOT explicitly."
    )
if missing_lcc_records:
    raise RuntimeError(f"LCC manifest/cache path mismatches: {missing_lcc_records[:5]}")
if len(lcc_rows) != len(src_manifest_rows):
    raise AssertionError("LCC candidate rows were silently lost")
if sum(int(row["label"]) == 0 for row in lcc_rows) < 1 or sum(int(row["label"]) == 1 for row in lcc_rows) < 1:
    raise RuntimeError("LCC external-dev set must contain Real and Attack")

lcc_manifest_out = STAGE0_DIR / "lcc_external_dev_manifest.csv"
with open(lcc_manifest_out, "w", newline="", encoding="utf-8") as f:
    fields = ["path", "label", "partition", "dataset", "subject_id", "attack_subtype", "file_sha256", "record_id", "status", "n_faces", "ambiguous_multiface"]
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    writer.writerows(lcc_rows)
selected_record_ids = {row["record_id"] for row in lcc_rows}
selected_lcc_records = {
    record_id: record_by_id[record_id]
    for record_id in selected_record_ids
    if record_id in record_by_id
}
if set(selected_lcc_records) != selected_record_ids:
    raise RuntimeError("LCC cache could not normalize every selected row by its record frame_id")
lcc_cache_payload = {
    "schema": "pad_micro_lcc_scrfd_cache_v1",
    "policy": cache_policy,
    "sampled_manifest_sha256": src_cache["sampled_manifest_sha256"],
    "records": selected_lcc_records,
}
lcc_cache_out = STAGE0_DIR / "lcc_scrfd_cache_micro_v1.json"
lcc_cache_out.write_text(json.dumps(lcc_cache_payload, indent=2, allow_nan=False) + "\n")

lcc_status_counts = Counter(str(row["status"]) for row in lcc_rows)
lcc_by_class = {
    str(class_id): {
        "candidates": sum(int(r["label"]) == class_id for r in lcc_rows),
        "valid_faces": sum(int(r["label"]) == class_id and r["status"] == "VALID" for r in lcc_rows),
        "failures": sum(int(r["label"]) == class_id and r["status"] != "VALID" for r in lcc_rows),
        "detector_coverage": (
            sum(int(r["label"]) == class_id and r["status"] == "VALID" for r in lcc_rows)
            / max(1, sum(int(r["label"]) == class_id for r in lcc_rows))
        ),
    } for class_id in (0, 1)
}
lcc_valid_paths = [str(r["path"]) for r in lcc_rows if str(r["status"]) == "VALID"]
lcc_valid_labels = [int(r["label"]) for r in lcc_rows if str(r["status"]) == "VALID"]
if not lcc_valid_paths or set(lcc_valid_labels) != {0, 1}:
    raise RuntimeError("Common LCC VALID set must be non-empty and contain Real plus Attack")
lcc_valid_paths_sha256 = ordered_path_fingerprint(lcc_valid_paths)
lcc_coverage = {
    "set_label": "LCC-FASD external-dev / cross-domain stress set",
    "candidate_n": len(lcc_rows),
    "real_n": sum(int(r["label"]) == 0 for r in lcc_rows),
    "attack_n": sum(int(r["label"]) == 1 for r in lcc_rows),
    "valid_face_n": sum(s == "VALID" for s in (r["status"] for r in lcc_rows)),
    "valid_paths_ordered_sha256": lcc_valid_paths_sha256,
    "valid_by_class": {
        str(class_id): sum(int(r["label"]) == class_id and r["status"] == "VALID" for r in lcc_rows)
        for class_id in (0, 1)
    },
    "detector_failure_n": sum(s != "VALID" for s in (r["status"] for r in lcc_rows)),
    "status_counts": dict(lcc_status_counts),
    "multi_face_n": sum(int(r["n_faces"]) > 1 for r in lcc_rows),
    "ambiguous_multiface_n": sum(bool(r["ambiguous_multiface"]) for r in lcc_rows),
    "by_class": lcc_by_class,
    "coverage": sum(s == "VALID" for s in (r["status"] for r in lcc_rows)) / max(1, len(lcc_rows)),
}
(STAGE0_DIR / "lcc_coverage_summary.json").write_text(json.dumps(lcc_coverage, indent=2, allow_nan=False) + "\n")
lcc_fingerprints = {
    "source_manifest_sha256": sha256_file(lcc_manifest_src) if lcc_manifest_src is not None else None,
    "source_cache_sha256": sha256_file(lcc_cache_src) if lcc_cache_src is not None else None,
    "source_protocol_snapshot_sha256": sha256_file(lcc_snapshot_src) if lcc_snapshot_src is not None else None,
    "source_detector_sha256": cache_policy["detector_sha256"],
    "source_sampled_manifest_sha256": src_cache["sampled_manifest_sha256"],
    "lcc_manifest_sha256": sha256_file(lcc_manifest_out),
    "lcc_cache_sha256": sha256_file(lcc_cache_out),
    "lcc_valid_path_fingerprint_sha256": lcc_valid_paths_sha256,
    "lcc_candidate_n": len(lcc_rows),
    "lcc_valid_face_n": len(lcc_valid_paths),
    "lcc_root_at_build": str(lcc_root),
}
(STAGE0_DIR / "lcc_resource_fingerprints.json").write_text(json.dumps(lcc_fingerprints, indent=2, allow_nan=False) + "\n")

# Optional lossless deterministic base crops for evaluation. Training always uses raw images
# so the frozen bbox jitter / mild augmentation policy remains available.
crop_cache_status = {"enabled": bool(BUILD_OPTIONAL_CROP_CACHE), "built": False, "reason": "disabled"}
if BUILD_OPTIONAL_CROP_CACHE:
    cache_dir = STAGE0_DIR / "crop_cache"
    index_path = cache_dir / "index.json"
    cache_dir.mkdir(parents=True, exist_ok=True)
    crop_index = {}
    total_bytes = 0
    image_specs = [("celeba", k, celeba_root / k, micro_bbox_payload["records"][k]) for k in micro_train + micro_val]
    for row in lcc_rows:
        if row["status"] == "VALID":
            rec = record_by_id[row["record_id"]]
            image_specs.append(("lcc", row["path"], lcc_root / row["path"], rec))
    try:
        for domain, key, image_path, rec in image_specs:
            bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
            if bgr is None:
                raise FileNotFoundError(image_path)
            crop = runtime_crop_bgr(bgr, rec["bbox_xyxy"], float(rec["crop_factor"]))
            rgb = resize_letterbox_rgb(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
            rel_name = hashlib.sha256(f"{domain}|{key}".encode()).hexdigest() + ".png"
            out_path = cache_dir / rel_name
            if not cv2.imwrite(str(out_path), cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)):
                raise RuntimeError(f"Could not save crop cache image: {out_path}")
            total_bytes += out_path.stat().st_size
            if total_bytes > OPTIONAL_CROP_CACHE_MAX_GB * (1024 ** 3):
                raise RuntimeError("Optional crop cache exceeded configured size guard")
            crop_index[f"{domain}::{key}"] = rel_name
        index_path.write_text(json.dumps(crop_index, indent=2, allow_nan=False) + "\n")
        crop_cache_status = {
            "enabled": True, "built": True, "count": len(crop_index),
            "bytes": total_bytes, "index": "crop_cache/index.json",
            "note": "Deterministic base crops only; never augmented crops."
        }
    except Exception as exc:
        shutil.rmtree(cache_dir, ignore_errors=True)
        crop_cache_status = {"enabled": True, "built": False, "reason": str(exc)}
        print("Optional crop cache skipped cleanly:", exc)

micro_manifest_sha = sha256_file(micro_npz_path)
resource_fingerprints = {
    "parent_e1_manifest_sha256": sha256_file(e1_manifest_path),
    "parent_e1_train_fingerprint": train_fp,
    "parent_e1_val_fingerprint": val_fp,
    "micro_manifest_sha256": micro_manifest_sha,
    "micro_train_keys_fingerprint": split_fingerprint(micro_train),
    "micro_val_keys_fingerprint": split_fingerprint(micro_val),
    "celeba_bbox_cache_sha256": sha256_file(bbox_out),
    "lcc_manifest_sha256": sha256_file(lcc_manifest_out),
    "lcc_cache_sha256": sha256_file(lcc_cache_out),
    "lcc_detector_sha256": cache_policy["detector_sha256"],
    "lcc_valid_path_fingerprint_sha256": lcc_valid_paths_sha256,
    "lcc_candidate_n": len(lcc_rows),
    "lcc_valid_face_n": len(lcc_valid_paths),
}
(STAGE0_DIR / "micro_resource_fingerprints.json").write_text(json.dumps(resource_fingerprints, indent=2, allow_nan=False) + "\n")
stage0_summary = {
    "schema": "pad_micro_stage0_summary_v1",
    "seed": int(SEED),
    "source_run_mode": run_config["run_mode"],
    "e1_manifest_name": e1_manifest_path.name,
    "e1_manifest_sha256": resource_fingerprints["parent_e1_manifest_sha256"],
    "e1_run_config_sha256": sha256_file(e1_run_config_path),
    "e1_bbox_source_sha256": micro_bbox_payload["source_sha256"],
    "micro_manifest_sha256": micro_manifest_sha,
    "micro_train_n": len(micro_train),
    "micro_val_n": len(micro_val),
    "micro_train_class_counts": micro_json["micro_train_class_counts"],
    "micro_val_class_counts": micro_json["micro_val_class_counts"],
    "micro_train_keys_fingerprint": micro_json["micro_train_keys_fingerprint"],
    "micro_val_keys_fingerprint": micro_json["micro_val_keys_fingerprint"],
    "bbox_status_audit": bbox_status_audit,
    "lcc_manifest_sha256": resource_fingerprints["lcc_manifest_sha256"],
    "lcc_cache_sha256": resource_fingerprints["lcc_cache_sha256"],
    "lcc_valid_path_fingerprint_sha256": lcc_valid_paths_sha256,
    "lcc_candidate_n": len(lcc_rows),
    "lcc_valid_face_n": len(lcc_valid_paths),
    "lcc_detector_sha256": cache_policy["detector_sha256"],
    "lcc_coverage": lcc_coverage,
    "crop_cache": crop_cache_status,
    "overlap_checks": micro_json["overlap_checks"],
}
(STAGE0_DIR / "stage0_summary.json").write_text(json.dumps(stage0_summary, indent=2, allow_nan=False) + "\n")

zip_path = Path(OUTPUT_ROOT) / "pad_micro_stage0_resources.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(STAGE0_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=Path("stage0") / path.relative_to(STAGE0_DIR))
print("\n=== WHAT TO ATTACH TO THE NEXT KAGGLE NOTEBOOK ===")
print("Stage 0 bundle ZIP:", zip_path)
print("Attach CelebA raw data + this Stage 0 bundle + LCC raw data.")
print("LCC detector policy:", json.dumps(cache_policy, sort_keys=True))
print("LCC source detector SHA:", cache_policy.get("detector_sha256"))
print("LCC standardized manifest/cache SHA:", lcc_fingerprints["lcc_manifest_sha256"], lcc_fingerprints["lcc_cache_sha256"])
print("LCC coverage detail:", json.dumps(lcc_coverage, indent=2, allow_nan=False))
print("Micro manifest SHA256:", micro_manifest_sha)


Resolving frozen E1 resources...
Selected CelebA bbox status/factor audit: {"accepted_status_factor_policy": {"FALLBACK_DET_FAIL": 1.5, "FALLBACK_SUSPICIOUS": 1.5, "VALID": 1.55}, "crop_factor_counts_by_status": {"FALLBACK_DET_FAIL": {"1.50": 70}, "FALLBACK_SUSPICIOUS": {"1.50": 3}, "VALID": {"1.55": 6927}}, "overall_status_counts": {"FALLBACK_DET_FAIL": 70, "FALLBACK_SUSPICIOUS": 3, "VALID": 6927}, "train_status_counts": {"FALLBACK_DET_FAIL": 47, "VALID": 4953}, "val_status_counts": {"FALLBACK_DET_FAIL": 23, "FALLBACK_SUSPICIOUS": 3, "VALID": 1974}}
Resolved all selected CelebA images: 7000 under /kaggle/input/datasets/attentionlayer241/celeba-spoof-for-face-antispoofing/CelebA_Spoof_/CelebA_Spoof
Frozen E1 manifest path/SHA: /kaggle/input/datasets/locngodev/e1-v5-3-artifacts/E1_v5_3_mnv3_small/data_protocol/celeba_spoof_preliminary_v5_3_edge_mnv3_small_seed42.npz 7799486ebcae0babe09e061579ac05db7c1dc61e10746be0755bcddc10b2df9e
Frozen E1 Train/Val fingerprints: 4ecbcee78758366dd454c0e